# Crime Volume Trends

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from dashboard.crime_dashboard_data import load_crime_dashboard_context
from dashboard.crime_classification import CANONICAL_CRIME_TYPES
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import plotly.graph_objects as go
warnings.filterwarnings('ignore')

context = load_crime_dashboard_context()
df = context["valid_time"].copy()
metadata = context["metadata"]

observed_crime_types = sorted(df["offense_category"].dropna().unique())
print("Observed analytical crime types:")
for value in observed_crime_types:
    print(f"  {value}")
assert set(observed_crime_types) == set(CANONICAL_CRIME_TYPES)
print("PASS: All observed crime types use the canonical v1.1 taxonomy.")

display(df.head())

CRIME_OUTPUT_DIR =  PROJECT_ROOT / "reports" / "crime"

TIME_COLUMN = "offense_date"
REPORT_TIME_COLUMN = "report_date_time"
EVENT_ID_COLUMN = "offense_id"
REPORT_ID_COLUMN = "report_number"
CATEGORY_COLUMN = "offense_category"

PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#545455"
PAPER_BG = "#111111"

Observed analytical crime types:
  crimes against persons
  crimes against property
  crimes against society / other
PASS: All observed crime types use the canonical v1.1 taxonomy.


,report_number,report_date_time,offense_id,offense_date,nibrs_group_a_b,nibrs_crime_against_category,offense_sub_category,shooting_type_group,block_address,latitude,...,nibrs_offense_code,census_block_2020,date,source_offense_category,classification_action,classification_reason,is_excluded_from_crime_analysis,event_group,event_importance_bin,mcpp_neighborhood
0,2024-944499,2024-11-08 17:33:31,60731686542,2024-09-08,a,property,burglary,-,22xx block of nw 57th st,47.670185,...,220,4701.2001,2024-09-08,property crime,retain,Default normalized source-category mapping.,False,crimes against property,crimes against property,ballard south
1,2024-255143,2024-09-08 13:05:22,58873361590,2024-09-08,a,property,extortion/fraud/forgery/bribery (includes bad ...,-,27xx block of s norman st,47.593315,...,26b,8900.2018,2024-09-08,all other,reclassify,NIBRS classifies Credit Card / Automated Telle...,False,crimes against property,crimes against property,madrona/leschi
2,2024-255181,2024-09-08 13:50:01,58873566302,2024-09-08,a,property,motor vehicle theft,-,2xx block of blaine st,47.634862,...,240,6800.1024,2024-09-08,property crime,retain,Default normalized source-category mapping.,False,crimes against property,crimes against property,queen anne
3,2024-256284,2024-09-09 12:55:47,58884571278,2024-09-08,a,person,assault offenses,-,redacted,NaN,...,13c,redacted,2024-09-08,all other,reclassify,NIBRS classifies Intimidation as a Crime Again...,False,crimes against persons,crimes against persons,lakecity
4,2024-257138,2024-09-10 09:01:11,58912501829,2024-09-08,a,property,larceny-theft,-,44xx block of ne 41st st,47.658413,...,23g,4101.3000,2024-09-08,property crime,retain,Default normalized source-category mapping.,False,crimes against property,crimes against property,sandpoint


In [2]:
TIME_COLUMN = "offense_date"
REPORT_TIME_COLUMN = "report_date_time"
EVENT_ID_COLUMN = "offense_id"
REPORT_ID_COLUMN = "report_number"

required_columns = [
    TIME_COLUMN,
    REPORT_TIME_COLUMN,
    EVENT_ID_COLUMN,
    REPORT_ID_COLUMN,
    "offense_category",
    "offense_sub_category",
    "nibrs_crime_against_category",
    "nibrs_group_a_b",
    "nibrs_offense_code_description",
    "nibrs_offense_code",
    "precinct",
    "sector",
    "beat",
    "neighborhood",
    "reporting_area",
    "latitude",
    "longitude",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

missing_columns

[]

In [ ]:
crime_df = df.copy()

crime_df[TIME_COLUMN] = pd.to_datetime(
    crime_df[TIME_COLUMN],
    errors="coerce",
)

crime_df[REPORT_TIME_COLUMN] = pd.to_datetime(
    crime_df[REPORT_TIME_COLUMN],
    errors="coerce",
)

crime_df[CATEGORY_COLUMN] = (
    crime_df[CATEGORY_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
)

crime_df = crime_df.dropna(subset=[TIME_COLUMN, EVENT_ID_COLUMN])

crime_df.shape
display(crime_df[[TIME_COLUMN, EVENT_ID_COLUMN, REPORT_ID_COLUMN]].head())
date_min = crime_df[TIME_COLUMN].min()
date_max = crime_df[TIME_COLUMN].max()

print(f"Start date of calls: {date_min} \nEnd date: {date_max}")

record_count = len(crime_df)
unique_crime_events = crime_df[EVENT_ID_COLUMN].nunique()
unique_report_records = crime_df[REPORT_ID_COLUMN].nunique()

print(f"Number of records: {record_count} \nNumber of unique report records: {unique_report_records} \nNumber of unique crime events: {unique_crime_events}")

In [ ]:
category_summary = (
    crime_df
    .groupby(CATEGORY_COLUMN, dropna=False)
    .agg(
        offense_count=(EVENT_ID_COLUMN, "size"),
        unique_offenses=(EVENT_ID_COLUMN, "nunique"),
        unique_reports=(REPORT_ID_COLUMN, "nunique"),
    )
    .reset_index()
    .sort_values("unique_offenses", ascending=False)
)

category_summary.head(30)

In [ ]:
selected_categories = (
    crime_df[CATEGORY_COLUMN]
    .dropna()
    .sort_values()
    .unique()
    .tolist()
)

selected_categories[:10], len(selected_categories)

In [ ]:
def build_crime_daily_volume(
    data: pd.DataFrame,
    selected_categories: list[str],
    time_column: str = TIME_COLUMN,
    event_id_column: str = EVENT_ID_COLUMN,
    category_column: str = CATEGORY_COLUMN,
) -> tuple[pd.DataFrame, dict]:
    working_df = data.copy()

    working_df[time_column] = pd.to_datetime(
        working_df[time_column],
        errors="coerce",
    )

    working_df[category_column] = (
        working_df[category_column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    working_df = working_df.dropna(
        subset=[
            time_column,
            event_id_column,
            category_column,
        ]
    )

    selected_categories = [
        str(category).strip().lower()
        for category in selected_categories
    ]

    filtered_df = working_df[
        working_df[category_column].isin(selected_categories)
    ].copy()

    if filtered_df.empty:
        raise ValueError("No records found for selected categories")

    filtered_df["date"] = filtered_df[time_column].dt.normalize()

    earliest_available_day = filtered_df["date"].min()
    latest_available_day = filtered_df["date"].max()

    # Match the calls-dashboard convention:
    # drop the earliest edge day because rolling snapshots may start mid-day.
    earliest_analysis_day = earliest_available_day + pd.Timedelta(days=1)

    past_year_start = latest_available_day - pd.Timedelta(days=364)

    plot_start_day = max(
        earliest_analysis_day,
        past_year_start,
    )

    plot_end_day = latest_available_day

    initial_view_start = max(
        plot_start_day,
        latest_available_day - pd.Timedelta(days=29),
    )

    plot_df = filtered_df[
        (filtered_df["date"] >= plot_start_day)
        & (filtered_df["date"] <= plot_end_day)
    ].copy()

    daily_volume = (
        plot_df
        .groupby("date")
        .agg(
            reported_offenses=(event_id_column, "nunique"),
            unique_reports=(REPORT_ID_COLUMN, "nunique"),
        )
        .reset_index()
    )

    full_date_range = pd.date_range(
        start=plot_start_day,
        end=plot_end_day,
        freq="D",
    )

    daily_volume = (
        daily_volume
        .set_index("date")
        .reindex(full_date_range)
        .rename_axis("date")
        .reset_index()
    )

    daily_volume["reported_offenses"] = (
        daily_volume["reported_offenses"]
        .fillna(0)
        .astype(int)
    )

    daily_volume["unique_reports"] = (
        daily_volume["unique_reports"]
        .fillna(0)
        .astype(int)
    )

    daily_volume["reported_offenses_7d_avg"] = (
        daily_volume["reported_offenses"]
        .rolling(window=7, min_periods=7)
        .mean()
    )

    date_context = {
        "earliest_available_day": earliest_available_day,
        "latest_available_day": latest_available_day,
        "earliest_analysis_day": earliest_analysis_day,
        "plot_start_day": plot_start_day,
        "plot_end_day": plot_end_day,
        "initial_view_start": initial_view_start,
        "selected_categories": selected_categories,
    }

    return daily_volume, date_context

In [ ]:
daily_volume, date_context = build_crime_daily_volume(
    crime_df,
    selected_categories=selected_categories,
)

daily_volume.head(), daily_volume.tail(), date_context

In [ ]:
selected_label = (
    "All selected categories"
    if len(selected_categories) > 1
    else selected_categories[0]
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=daily_volume["date"],
        y=daily_volume["reported_offenses"],
        name="Daily reported offenses",
        opacity=0.35,
        hovertemplate=(
            "<b>%{x|%b %d, %Y}</b><br>"
            "Reported offenses: %{y:,}<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=daily_volume["date"],
        y=daily_volume["reported_offenses_7d_avg"],
        mode="lines",
        name="7-day average",
        line=dict(width=3),
        hovertemplate=(
            "<b>%{x|%b %d, %Y}</b><br>"
            "7-day avg: %{y:.1f}<extra></extra>"
        ),
    )
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title=(
        "Daily Reported Crime Offenses"
        f"<br><sup>Category: {selected_label}</sup>"
    ),
    xaxis_title="Offense Date",
    yaxis_title="Reported Offenses",
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    height=650,
    margin=dict(l=70, r=40, t=90, b=80),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
)

fig.update_xaxes(
    range=[
        date_context["initial_view_start"],
        date_context["plot_end_day"],
    ],
    rangeslider=dict(visible=True),
)

fig.show()

In [ ]:
selected_categories = [CANONICAL_CRIME_TYPES[0]]

daily_volume, date_context = build_crime_daily_volume(
    crime_df,
    selected_categories=selected_categories,
)

daily_volume.head(), daily_volume.tail(), date_context

In [ ]:
daily_by_category = (
    crime_df
    .dropna(subset=[TIME_COLUMN, CATEGORY_COLUMN])
    .assign(date=lambda x: x[TIME_COLUMN].dt.normalize())
    .groupby(["date", CATEGORY_COLUMN])
    .agg(reported_offenses=(EVENT_ID_COLUMN, "nunique"))
    .reset_index()
)

category_daily_stats = (
    daily_by_category
    .groupby(CATEGORY_COLUMN)
    .agg(
        active_days=("date", "nunique"),
        mean_daily_offenses=("reported_offenses", "mean"),
        median_daily_offenses=("reported_offenses", "median"),
        max_daily_offenses=("reported_offenses", "max"),
    )
    .reset_index()
    .sort_values("mean_daily_offenses", ascending=False)
)

category_daily_stats.head(30)